# Transformer Encoder with CTC

The second of the four architectures. A Transformer encoder over per-frame pose features, trained with CTC loss and decoded with beam search.

This isolates one variable against the Conformer in `01-bilstm-and-conformer-ctc.ipynb`: both use self-attention over the same inputs with the same loss, but the Conformer adds a convolutional module for local frame-to-frame motion and this one does not.

**Best dev WER: 36.34%**, against 66.62% for the recurrent baseline and 13.04% for the Conformer.

One caveat to carry into the comparison: that best score arrives at **epoch 30 of 30**, the final epoch, so training ended while the model was still improving. 36.34% is a ceiling on what was measured, not a converged result.

# 1. Data Acquisition (Download & Unzip)

Downloading the Isharah dataset from Google Drive and unzipping it into the local environment for faster access.

In [1]:
# 1. Install gdown (usually pre-installed, but just in case)
!pip install -q gdown

# 2. Download using the ID from your link
# The ID is the part between /d/ and /view: 1d9BcEjNkyLsSdE_tzyPiUTuSFbbpuQ8M
!gdown 1d9BcEjNkyLsSdE_tzyPiUTuSFbbpuQ8M -O train_data.zip

# 3. Now try unzipping again
!unzip -q train_data.zip -d ./isharah_data/

# 4. Check if it worked by listing the first few files
import os
print("Files found:", len(os.listdir('./isharah_data/')))

Downloading...
From (original): https://drive.google.com/uc?id=1d9BcEjNkyLsSdE_tzyPiUTuSFbbpuQ8M
From (redirected): https://drive.google.com/uc?id=1d9BcEjNkyLsSdE_tzyPiUTuSFbbpuQ8M&confirm=t&uuid=a55d0f8e-12a9-4657-8620-96fb7ee723e0
To: /content/train_data.zip
100% 1.68G/1.68G [00:17<00:00, 94.9MB/s]
Files found: 4


In [2]:
import os

root_dir = './isharah_data/'
items = os.listdir(root_dir)

print(f"--- Items in {root_dir} ---")
for item in items:
    path = os.path.join(root_dir, item)
    if os.path.isdir(path):
        # If it's a folder, count how many files are inside
        num_files = len(os.listdir(path))
        print(f" Folder: {item} (Contains {num_files} files)")
    else:
        # If it's a file, show its size
        size_mb = os.path.getsize(path) / (1024 * 1024)
        print(f" File:   {item} ({size_mb:.2f} MB)")

--- Items in ./isharah_data/ ---
 File:   train.csv (0.46 MB)
 Folder: __MACOSX (Contains 3 files)
 File:   pose_data_isharah1000_hands_lips_body_May12.pkl (3075.77 MB)
 File:   dev.csv (0.04 MB)


# 2. Data Inspection

Here we inspect the directory structure and verify that the .pkl and .csv files are loaded correctly. This step ensures the data integrity before processing.

In [3]:
import pandas as pd
import pickle
import os

root_dir = './isharah_data/'
pkl_path = os.path.join(root_dir, 'pose_data_isharah1000_hands_lips_body_May12.pkl')
train_csv_path = os.path.join(root_dir, 'train.csv')

# --- 1. Inspect the CSV ---
print("--- CSV Structure (First 3 rows) ---")
try:
    df = pd.read_csv(train_csv_path)
    print(df.head(3))
    print(f"\nColumns: {list(df.columns)}")
except Exception as e:
    print(f"Error reading CSV: {e}")

# --- 2. Inspect the Pickle (RAM Intensive) ---
print("\n--- Pickle Structure ---")
print("Loading 3GB pickle file... (This uses ~3GB RAM)")

try:
    with open(pkl_path, 'rb') as f:
        pose_data = pickle.load(f)

    print("Pickle loaded successfully!")
    print(f"Type of data: {type(pose_data)}")

    if isinstance(pose_data, dict):
        # Print first key and shape of its value
        first_key = list(pose_data.keys())[0]
        first_val = pose_data[first_key]
        print(f"\nSample Key: {first_key}")

        # Check if it's a numpy array or list
        if hasattr(first_val, 'shape'):
            print(f"Value Shape: {first_val.shape}")
        elif isinstance(first_val, list):
            print(f"Value is a list of length {len(first_val)}")
            if len(first_val) > 0:
                 print(f"First frame data: {first_val[0]}")
        else:
            print(f"Value type: {type(first_val)}")

        # Check if the CSV filenames match the Pickle keys
        # Assuming the first column of CSV is the ID
        csv_id = str(df.iloc[0, 0])
        if csv_id in pose_data:
            print(f"\ MATCH CONFIRMED: CSV ID '{csv_id}' found in Pickle keys.")
        else:
            print(f"\ MISMATCH: CSV ID '{csv_id}' NOT found in Pickle keys.")
            print(f"First 5 Pickle Keys: {list(pose_data.keys())[:5]}")

    # Free up memory if needed, or keep 'pose_data' if you have enough RAM
    # del pose_data

except Exception as e:
    print(f"Error reading Pickle: {e}")

<>:49: SyntaxWarning: invalid escape sequence '\ '
<>:51: SyntaxWarning: invalid escape sequence '\ '
<>:49: SyntaxWarning: invalid escape sequence '\ '
<>:51: SyntaxWarning: invalid escape sequence '\ '
/tmp/ipython-input-2325292338.py:49: SyntaxWarning: invalid escape sequence '\ '
  print(f"\ MATCH CONFIRMED: CSV ID '{csv_id}' found in Pickle keys.")
/tmp/ipython-input-2325292338.py:51: SyntaxWarning: invalid escape sequence '\ '
  print(f"\ MISMATCH: CSV ID '{csv_id}' NOT found in Pickle keys.")


--- CSV Structure (First 3 rows) ---
        id               gloss
0  00_0001             سوال هو
1  00_0002   هو معلم لغه اشاره
2  00_0003  استفهام هو معلم هو

Columns: ['id', 'gloss']

--- Pickle Structure ---
Loading 3GB pickle file... (This uses ~3GB RAM)


/tmp/ipython-input-2325292338.py:24: DeprecationWarning: numpy.core.numeric is deprecated and has been renamed to numpy._core.numeric. The numpy._core namespace contains private NumPy internals and its use is discouraged, as NumPy internals can change without warning in any release. In practice, most real-world usage of numpy.core is to access functionality in the public NumPy API. If that is the case, use the public NumPy API. If not, you are using NumPy internals. If you would still like to access an internal attribute, use numpy._core.numeric._frombuffer.
  pose_data = pickle.load(f)


Pickle loaded successfully!
Type of data: <class 'dict'>

Sample Key: 00_0001
Value type: <class 'dict'>
\ MATCH CONFIRMED: CSV ID '00_0001' found in Pickle keys.


In [4]:
import pandas as pd
import numpy as np
from collections import Counter
import os

class Vocabulary:
    def __init__(self, csv_path):
        self.df = pd.read_csv(csv_path)
        self.idx2word = {}
        self.word2idx = {}
        self.build_vocab()

    def build_vocab(self):
        all_glosses = self.df['gloss'].tolist()
        word_counts = Counter()

        for gloss in all_glosses:
            words = str(gloss).split() # Split by space
            word_counts.update(words)

        # 0 is usually reserved for padding, 1 for CTC blank
        self.idx2word = {0: '<pad>', 1: '<blank>'}
        self.word2idx = {'<pad>': 0, '<blank>': 1}

        idx = 2
        for word, _ in word_counts.items():
            self.word2idx[word] = idx
            self.idx2word[idx] = word
            idx += 1

        print(f"Vocabulary built! Unique words: {len(self.word2idx)} (including special tokens)")

    def text_to_indices(self, text):
        words = str(text).split()
        return [self.word2idx[w] for w in words if w in self.word2idx]

    def indices_to_text(self, indices):
        words = []
        for idx in indices:
            # CTC decoding often produces 1 (blank) or repeated characters
            if idx in self.idx2word and idx != 1 and idx != 0:
                words.append(self.idx2word[idx])
        return " ".join(words)

# Initialize it
# Make sure the path exists
if os.path.exists('./isharah_data/train.csv'):
    vocab = Vocabulary('./isharah_data/train.csv')

    # Test it
    sample_text = "سوال هو"
    indices = vocab.text_to_indices(sample_text)
    print(f"Original: {sample_text}")
    print(f"Indices: {indices}")
else:
    print(" Error: './isharah_data/train.csv' not found. Did you upload the CSVs?")

Vocabulary built! Unique words: 682 (including special tokens)
Original: سوال هو
Indices: [2, 3]


# 3. Vocabulary Builder

We define a Vocabulary class to map sign language glosses (words) to numerical indices. This is essential for converting the text labels into a format the model can learn.

In [5]:
import torch
import numpy as np
import pandas as pd
import random
import math
from torch.utils.data import Dataset, DataLoader
from scipy import interpolate # Needed for time warping
import pickle # Added
import os     # Added
from collections import Counter # Added for Vocabulary class

# Re-load pose_data here to ensure it's defined
root_dir = './isharah_data/'
pkl_path = os.path.join(root_dir, 'pose_data_isharah1000_hands_lips_body_May12.pkl')

try:
    with open(pkl_path, 'rb') as f:
        pose_data = pickle.load(f)
    print("pose_data reloaded successfully within this cell.")
except Exception as e:
    print(f"Error reloading pose_data: {e}")

# --- Vocabulary Class (copied from K2wZlZpyWQvt) ---
class Vocabulary:
    def __init__(self, csv_path):
        self.df = pd.read_csv(csv_path)
        self.idx2word = {}
        self.word2idx = {}
        self.build_vocab()

    def build_vocab(self):
        all_glosses = self.df['gloss'].tolist()
        word_counts = Counter()

        for gloss in all_glosses:
            words = str(gloss).split() # Split by space
            word_counts.update(words)

        # 0 is usually reserved for padding, 1 for CTC blank
        self.idx2word = {0: '<pad>', 1: '<blank>'}
        self.word2idx = {'<pad>': 0, '<blank>': 1}

        idx = 2
        for word, _ in word_counts.items():
            self.word2idx[word] = idx
            self.idx2word[idx] = word
            idx += 1

        print(f"Vocabulary built! Unique words: {len(self.word2idx)} (including special tokens)")

    def text_to_indices(self, text):
        """Converts 'سوال هو' -> [23, 45]"""
        words = str(text).split()
        return [self.word2idx[w] for w in words if w in self.word2idx]

    def indices_to_text(self, indices):
        """Converts [23, 45] -> 'سوال هو'"""
        words = []
        for idx in indices:
            # CTC decoding often produces 1 (blank) or repeated characters
            if idx in self.idx2word and idx != 1 and idx != 0:
                words.append(self.idx2word[idx])
        return " ".join(words)

TRAIN_CSV = './isharah_data/train.csv'
if os.path.exists(TRAIN_CSV):
    vocab = Vocabulary(TRAIN_CSV)
else:
    print(f"Error: '{TRAIN_CSV}' not found.")




/tmp/ipython-input-1821936985.py:18: DeprecationWarning: numpy.core.numeric is deprecated and has been renamed to numpy._core.numeric. The numpy._core namespace contains private NumPy internals and its use is discouraged, as NumPy internals can change without warning in any release. In practice, most real-world usage of numpy.core is to access functionality in the public NumPy API. If that is the case, use the public NumPy API. If not, you are using NumPy internals. If you would still like to access an internal attribute, use numpy._core.numeric._frombuffer.
  pose_data = pickle.load(f)


pose_data reloaded successfully within this cell.
Vocabulary built! Unique words: 682 (including special tokens)
Dataset initialized (Augment=True). Valid samples: 9500
Dataset initialized (Augment=False). Valid samples: 949
 DataLoaders ready. Training set will use Time Warping & Rotation.


#4. Dataset & DataLoader Definition

We define the custom IsharahDataset class, which handles loading the pose keypoints and normalizing inputs. We also define the DataLoader to batch the data efficiently during training.

In [ ]:
class IsharahDataset(Dataset):
    def __init__(self, pose_dict, csv_path, vocab, augment=False):
        """
        pose_dict: Dictionary {video_id: {'keypoints': numpy_array}}
        csv_path: Path to the CSV file (train.csv or dev.csv)
        vocab: Vocabulary object for text-to-index conversion
        augment: Boolean. If True, applies random transformations.
        """
        self.pose_dict = pose_dict
        self.df = pd.read_csv(csv_path)
        self.vocab = vocab
        self.augment = augment

        # Filter: Only keep IDs that exist in BOTH the CSV and the loaded Pickle file
        self.valid_ids = [vid for vid in self.df['id'] if vid in self.pose_dict]

        print(f"Dataset initialized (Augment={self.augment}). Valid samples: {len(self.valid_ids)}")

    def __len__(self):
        return len(self.valid_ids)

    # --- Augmentation Strategies ---
    def augment_time_warp(self, data):
        """
        Randomly speeds up or slows down the sign (Temporal Jitter).
        Resamples the frames to be 80% to 120% of original length.
        """
        # Data shape is (Time, Vertices, Channels)
        T, V, C = data.shape

        # 1. flatten to (T, Features) for easier interpolation
        flat_data = data.reshape(T, -1)

        # 2. Generate new time axis
        speed_factor = random.uniform(0.8, 1.2) # Speed up or slow down
        new_T = int(T * speed_factor)

        # If too short, skip warping
        if new_T < 5:
            return data

        x_old = np.linspace(0, 1, T)
        x_new = np.linspace(0, 1, new_T)

        # 3. Interpolate
        f = interpolate.interp1d(x_old, flat_data, axis=0, kind='linear', fill_value="extrapolate")
        new_data = f(x_new)

        # 4. Reshape back
        return new_data.reshape(new_T, V, C)

    def augment_geometric(self, data):
        """
        Applies Rotation, Scaling, and Noise to the spatial coordinates.
        Data shape: (Time, Vertices, 2)
        """
        # 1. Random Rotation (±13 degrees)
        theta = random.uniform(-0.2, 0.2)
        c, s = np.cos(theta), np.sin(theta)
        rot_mat = np.array([[c, -s], [s, c]])

        # 2. Random Scale (±15%)
        scale = random.uniform(0.85, 1.15)

        # 3. Gaussian Noise (Simulate sensor jitter)
        noise = np.random.normal(0, 0.001, data.shape) # Small noise

        # Apply Rotation
        # Reshape to (N, 2) to multiply with matrix
        original_shape = data.shape
        flat_data = data.reshape(-1, 2)

        # Rotate
        flat_data = np.dot(flat_data, rot_mat)

        # Scale
        flat_data = flat_data * scale

        # Reshape back and add Noise
        data = flat_data.reshape(original_shape) + noise

        return data

    def __getitem__(self, idx):
        # 1. Get ID
        vid_id = self.valid_ids[idx]

        # 2. Get Text Label (Gloss) & Convert to Indices
        gloss_text = self.df[self.df['id'] == vid_id].iloc[0]['gloss']
        y = torch.tensor(self.vocab.text_to_indices(gloss_text), dtype=torch.long)

        # 3. Get Pose Data
        # Shape: (Time, 86, 2)
        pose_data_item = self.pose_dict[vid_id]['keypoints'].copy()

        # 4. Normalization (CRITICAL: Do this BEFORE augmentation usually, or after.
        # Here we center first so rotation works around the center)
        # Calculate mean across vertices to center the pose
        center = pose_data_item.mean(axis=1, keepdims=True)
        pose_data_item = pose_data_item - center

        # 5. Apply Augmentations (Only if training)
        if self.augment:
            # Randomly decide to apply Time Warp (50% chance)
            if random.random() < 0.5:
                pose_data_item = self.augment_time_warp(pose_data_item)

            # Always apply geometric noise/rotation in training
            pose_data_item = self.augment_geometric(pose_data_item)

        # 6. Convert to Tensor
        x = torch.tensor(pose_data_item, dtype=torch.float32)

        return x, y

# --- Custom Collate Function for DataLoader ---
def padding_collate_fn(batch):
    # batch is a list of (data, label) tuples
    # Each data item (x) is a tensor of shape (Time, Vertices, Channels)
    # Each label item (y) is a tensor of shape (LabelSequenceLength)

    # Separate data and labels
    xs, ys = zip(*batch)

    # Pad sequences to the longest sequence in the batch
    # Find the maximum time dimension (sequence length)
    max_seq_len = max(x.shape[0] for x in xs)

    # Pad x (pose data)
    padded_xs = []
    for x in xs:
        padding_needed = max_seq_len - x.shape[0]
        # Pad with zeros along the time dimension
        padded_x = torch.nn.functional.pad(x, (0, 0, 0, 0, 0, padding_needed), 'constant', 0)
        padded_xs.append(padded_x)
    padded_xs = torch.stack(padded_xs)

    # Pad y (labels)
    max_label_len = max(y.shape[0] for y in ys)
    padded_ys = []
    for y in ys:
        padding_needed = max_label_len - y.shape[0]
        padded_y = torch.nn.functional.pad(y, (0, padding_needed), 'constant', 0)
        padded_ys.append(padded_y)
    padded_ys = torch.stack(padded_ys)

    # Create length tensors for CTC Loss (required by some models)
    x_lengths = torch.tensor([x.shape[0] for x in xs], dtype=torch.long)
    y_lengths = torch.tensor([y.shape[0] for y in ys], dtype=torch.long)

    return padded_xs, x_lengths, padded_ys, y_lengths

# --- Initialize Loaders ---

# Make sure your paths are correct
DEV_CSV = './isharah_data/dev.csv'

# Train Loader: Augment = True
train_dataset = IsharahDataset(pose_data, TRAIN_CSV, vocab, augment=True)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, collate_fn=padding_collate_fn)

# Dev Loader: Augment = False (Evaluation must be clean)
dev_dataset = IsharahDataset(pose_data, DEV_CSV, vocab, augment=False)
dev_loader = DataLoader(dev_dataset, batch_size=32, shuffle=False, collate_fn=padding_collate_fn)



In [6]:
# 1. Initialize Dataset
train_dataset = IsharahDataset(pose_data, './isharah_data/train.csv', vocab)

# 2. Initialize DataLoader
# Batch size 32 is standard. Num_workers=0 avoids multiprocessing errors in Colab.
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True,
                          collate_fn=padding_collate_fn)

# 3. Fetch one batch to verify shapes
data_iter = iter(train_loader)
batch_x, batch_y, x_lens, y_lens = next(data_iter)

print("\n--- Batch Verification ---")
print(f"Batch X Shape: {batch_x.shape}") # Should be (32, Max_Frames, 86, 2)
print(f"Batch Y Shape: {batch_y.shape}") # Should be (32, Max_Words)
print(f"X Lengths: {x_lens[:5]}")        # Real frame counts before padding
print(f"Y Lengths: {y_lens[:5]}")        # Real word counts

Dataset initialized (Augment=False). Valid samples: 9500

--- Batch Verification ---
Batch X Shape: torch.Size([32, 427, 86, 2])
Batch Y Shape: torch.Size([32])
X Lengths: tensor([[  3,  37,  43,  44,   2,   0,   0,   0],
        [ 24,  11,  25,  26,   0,   0,   0,   0],
        [409, 137,  96,  33,  59,   0,   0,   0],
        [181, 343, 342,   0,   0,   0,   0,   0],
        [  7,   3, 316, 435, 110,   0,   0,   0]])
Y Lengths: tensor([5, 4, 5, 3, 5])


# 5. Model Architecture: Sign Language Transformer

This section defines the Transformer-based architecture. It consists of a Linear Embedding layer, Positional Encoding to retain temporal information, and a Transformer Encoder to capture dependencies between frames.

In [7]:
import torch
import torch.nn as nn
import math
import os
import pandas as pd
import numpy as np
from torch.utils.data import DataLoader
try:
    import jiwer
except ImportError:
    !pip install -q jiwer
    import jiwer

# --- 1. Re-Define the Model (With Fixed Input Dim) ---
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000, dropout=0.1):
        super().__init__()
        self.dropout = nn.Dropout(p=dropout)
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0)
        self.register_buffer('pe', pe)

    def forward(self, x):
        x = x + self.pe[:, :x.size(1), :]
        return self.dropout(x)

class SignLanguageTransformer(nn.Module):
    def __init__(self, vocab_size, input_dim=172, d_model=512, nhead=8, num_layers=4):
        super().__init__()
        self.embedding = nn.Linear(input_dim, d_model)
        self.bn = nn.BatchNorm1d(d_model) # Normalization helps convergence
        self.pos_encoder = PositionalEncoding(d_model)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead, dim_feedforward=2048, dropout=0.1, batch_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.classifier = nn.Linear(d_model, vocab_size)

    def forward(self, x, src_key_padding_mask=None):
        # 1. Project & Normalize
        # x: [Batch, Frames, 86, 2] -> Flattened to [Batch, Frames, 172] BEFORE being passed in
        # We need to reshape x if it's not already flattened, but the training loop does this.
        # So input_dim=172 matches the flattened input.

        x = self.embedding(x)
        x = x.permute(0, 2, 1) # Swap for BatchNorm (Batch, Feat, Time)
        x = self.bn(x)
        x = x.permute(0, 2, 1) # Swap back (Batch, Time, Feat)

        # 2. Add Position Info
        x = self.pos_encoder(x)

        # 3. Transformer Layers
        # src_key_padding_mask allows model to ignore padding zeros
        output = self.transformer(x, src_key_padding_mask=src_key_padding_mask)

        # 4. Predict
        logits = self.classifier(output)
        return torch.nn.functional.log_softmax(logits, dim=2)

# --- Helper for Masking ---
def create_mask(x_lens, max_len):
    """
    Generates a boolean mask for Transformer Encoder, indicating padding elements.
    True indicates a masked (ignored) element.
    """
    # Ensure x_lens is on the same device as the mask operation
    # The device should be passed or inferred if possible for robustness.
    device = x_lens.device
    # max_len is `t` from the batch.size()
    return (torch.arange(max_len, device=device).expand(len(x_lens), max_len) >= x_lens.unsqueeze(1))


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 56.2 MB/s eta 0:00:00


# 6. Training Configuration & Utilities

Setting up the hyperparameters, optimizer (AdamW), and the CTC Loss function. We also define a helper function decode_ctc to calculate the Word Error Rate (WER) during validation.

In [13]:
import torch
import torch.nn as nn
import torch.optim as optim
from tqdm import tqdm
import os
import jiwer
import numpy as np
from google.colab import drive

# --- 1. Setup Drive & Save Path ---
drive.mount('/content/drive')
save_dir = '/content/drive/MyDrive/Isharah_Transformer_Models'
os.makedirs(save_dir, exist_ok=True)
best_model_path = os.path.join(save_dir, 'best_transformer_ctc.pth')
print(f" Models will be saved to: {save_dir}")

# --- 2. Helper: Decode CTC Output to Text ---
# (We need this to calculate WER
def decode_ctc(log_probs, vocab):
    preds = torch.argmax(log_probs, dim=2).detach().cpu().numpy()
    sentences = []
    for sequence in preds:
        indices = []
        prev = -1
        for char in sequence:
            if char != prev and char != 1: # 1 is Blank
                if char != 0: # 0 is Pad
                    indices.append(char)
            prev = char
        # Convert IDs to Words
        if hasattr(vocab, 'indices_to_text'):
            sentences.append(vocab.indices_to_text(indices))
        else: # Fallback for dict (this path won't be taken with current setup)
            words = [vocab.id2token[i] for i in indices if i in vocab.id2token]
            sentences.append(" ".join(words))
    return sentences

# --- 4. Setup Training ---
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Check Input Dimension from Data
sample_batch = next(iter(train_loader))
input_dim = sample_batch[0].shape[-1] * sample_batch[0].shape[-2] # Flatten v*c
# Or if already flattened in loader, just shape[-1]
if len(sample_batch[0].shape) == 4:
    input_dim = sample_batch[0].shape[2] * sample_batch[0].shape[3]

VOCAB_SIZE = len(vocab.idx2word) # Correctly use vocab.idx2word for size
model = SignLanguageTransformer(vocab_size=VOCAB_SIZE, input_dim=input_dim).to(device) # No need for +1 as idx2word includes special tokens

optimizer = optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-3)
criterion = nn.CTCLoss(blank=1, zero_infinity=True)

print(" Starting Training ( WER + Drive Save)...")

# --- 5. Training Loop ---
num_epochs = 30
best_wer = float('inf')

for epoch in range(num_epochs):
    model.train()
    total_loss = 0
    loop = tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs}")

    for batch_idx, (padded_x, x_lengths, padded_y, y_lengths) in enumerate(loop):
        padded_x, padded_y = padded_x.to(device), padded_y.to(device)
        x_lengths, y_lengths = x_lengths.to(device), y_lengths.to(device) # Move lengths to device

        # Flatten [B, T, V, C] -> [B, T, F]
        b, t, v, c = padded_x.size()
        flattened_x = padded_x.view(b, t, v*c)

        # Mask
        mask = torch.arange(t).expand(len(x_lengths), t).to(device) >= x_lengths.unsqueeze(1)

        optimizer.zero_grad()
        log_probs = model(flattened_x, src_key_padding_mask=mask)

        # CTC Loss
        loss = criterion(log_probs.permute(1, 0, 2), padded_y, x_lengths, y_lengths)

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=0.5)
        optimizer.step()

        total_loss += loss.item()
        loop.set_postfix(loss=loss.item())

    avg_loss = total_loss / len(train_loader)

    # --- EVALUATION (Calculate WER) ---
    model.eval()
    val_preds = []
    val_refs = []
    with torch.no_grad():
        for x, x_len, y, y_len in dev_loader:
            x = x.to(device)
            x_len = x_len.to(device) # Move lengths to device
            y_len = y_len.to(device) # Move lengths to device

            b, t, v, c = x.size()
            x = x.view(b, t, v*c)
            mask = torch.arange(t).expand(len(x_len), t).to(device) >= x_len.unsqueeze(1)

            logits = model(x, src_key_padding_mask=mask)

            # Decode using our helper
            # The `vocab if 'vocab' in globals() ...` part ensures the vocab object is passed
            decoded_preds = decode_ctc(logits, vocab)
            val_preds.extend(decoded_preds)

            # Decode Truth
            y_np = y.cpu().numpy()
            for i in range(len(y_np)):
                real_len = y_len[i].item()
                indices = y_np[i][:real_len]
                # Correctly use vocab.idx2word for ground truth decoding
                words = [vocab.idx2word[idx] for idx in indices if idx in vocab.idx2word]
                val_refs.append(" ".join(words))

    # Compute Metric
    dev_wer = jiwer.wer(val_refs, val_preds)
    print(f"Epoch {epoch+1} | Loss: {avg_loss:.4f} | Dev WER: {dev_wer:.4f}")

    # --- SAVE TO DRIVE ---
    if dev_wer < best_wer:
        best_wer = dev_wer
        torch.save(model.state_dict(), best_model_path)
        print(f" New Best Model Saved to Drive! (WER: {best_wer:.4f})")

    # Regular Checkpoint
    if (epoch + 1) % 5 == 0:
        torch.save(model.state_dict(), os.path.join(save_dir, f'epoch_{epoch+1}.pth'))

print(" Done.")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
 Models will be saved to: /content/drive/MyDrive/Isharah_Transformer_Models
 Starting Training ( WER + Drive Save)...


Epoch 1/30: 100%|██████████| 297/297 [02:50<00:00,  1.74it/s, loss=5.67]
/usr/local/lib/python3.12/dist-packages/torch/nn/modules/transformer.py:515: UserWarning: The PyTorch API of nested tensors is in prototype stage and will change in the near future. We recommend specifying layout=torch.jagged when constructing a nested tensor, as this layout receives active development, has better operator coverage, and works with torch.compile. (Triggered internally at /pytorch/aten/src/ATen/NestedTensorImpl.cpp:178.)
  output = torch._nested_tensor_from_mask(


Epoch 1 | Loss: 7.6683 | Dev WER: 0.9879
 New Best Model Saved to Drive! (WER: 0.9879)


Epoch 2/30: 100%|██████████| 297/297 [02:50<00:00,  1.74it/s, loss=4.27]


Epoch 2 | Loss: 4.9578 | Dev WER: 0.9877
 New Best Model Saved to Drive! (WER: 0.9877)


...

Epoch 28 | Loss: 0.6490 | Dev WER: 0.4155


Epoch 29/30: 100%|██████████| 297/297 [02:50<00:00,  1.75it/s, loss=0.657]


Epoch 29 | Loss: 0.6219 | Dev WER: 0.4109


Epoch 30/30: 100%|██████████| 297/297 [02:50<00:00,  1.74it/s, loss=0.885]


Epoch 30 | Loss: 0.6135 | Dev WER: 0.3634
 New Best Model Saved to Drive! (WER: 0.3634)
 Done.


In [27]:
import torch
import jiwer
import torch.nn.functional as F

def ctc_beam_search_decoder(log_probs_tensor, vocab, beam_width=5, blank_idx=1):

    log_probs = log_probs_tensor.cpu().detach()
    T, V = log_probs.shape

    beams = [(0.0, [])]

    for t in range(T):

        step_log_probs = log_probs[t]
        top_k_probs, top_k_indices = torch.topk(step_log_probs, beam_width)

        new_beams = []
        for score, seq in beams:
            for i in range(beam_width):
                prob = top_k_probs[i].item()
                idx = top_k_indices[i].item()

                new_beams.append((score + prob, seq + [idx]))

        new_beams.sort(key=lambda x: x[0], reverse=True)
        beams = new_beams[:beam_width]

    best_score, best_raw_indices = beams[0]

    decoded_indices = []
    prev_idx = -1

    for idx in best_raw_indices:
        if idx != prev_idx and idx != blank_idx:
            decoded_indices.append(idx)
        prev_idx = idx

    words = [vocab.idx2word[i] for i in decoded_indices if i in vocab.idx2word]
    return " ".join(words)

print(f"--- Starting Evaluation with Beam Search (Width=5) ---")

model.eval()
preds = []
refs = []

BLANK_IDX = 1

with torch.no_grad():
    for batch_idx, (x, x_len, y, y_len) in enumerate(dev_loader):
        x = x.to(device)

        b, t, v, c = x.size()
        flattened_x = x.view(b, t, v*c)

        mask = torch.arange(t).expand(len(x_len), t).to(device) >= x_len.to(device).unsqueeze(1)

        log_probs = model(flattened_x, src_key_padding_mask=mask)
        # log_probs shape: [Batch, Time, Vocab]

        for i in range(b):
            real_length = x_len[i].item()

            sequence_log_probs = log_probs[i, :real_length, :]

            pred_text = ctc_beam_search_decoder(sequence_log_probs, vocab, beam_width=5, blank_idx=BLANK_IDX)
            preds.append(pred_text)

            y_np = y.cpu().numpy()
            target_len = y_len[i].item()
            target_indices = y_np[i][:target_len]

            target_words = [vocab.idx2word[idx] for idx in target_indices if idx in vocab.idx2word]
            refs.append(" ".join(target_words))

if len(preds) > 0:
    wer_score = jiwer.wer(refs, preds)
    print(f"\nEvaluation Complete!")
    print(f"Beam Search WER: {wer_score:.4f} ({wer_score*100:.2f}%)")

    print("\n--- Sample Predictions ---")
    for i in range(min(5, len(preds))):
        print(f"Ref : {refs[i]}")
        print(f"Pred: {preds[i]}")
        print("-" * 30)
else:
    print("Warning: No predictions were made. Check your DataLoader.")

--- Starting Evaluation with Beam Search (Width=5) ---

Evaluation Complete!
Beam Search WER: 0.3634 (36.34%)

--- Sample Predictions ---
Ref : سوال هو
Pred: سوال هو
------------------------------
Ref : هو معلم لغه اشاره
Pred: هو هو لغه اشاره
------------------------------
Ref : استفهام هو معلم هو
Pred: استفهام استفهام هو
------------------------------
Ref : هو معلم لا انا مدرسه
Pred: هو معلم لا انا
------------------------------
Ref : هو سوال
Pred: هو سوال سوال
------------------------------


# 7. Error Analysis

In [33]:
import pandas as pd
import jiwer
from collections import Counter

print("--- Detailed Error Analysis ---")

# Initialize lists to store error details
all_deletions = []      # Words that were in the reference but missing in prediction
length_diffs = []       # Difference in sentence length (Prediction - Reference)
error_samples = []      # List to store specific examples of errors

# Loop through all validation samples
for i in range(len(refs)):
    ref_sentence = refs[i]
    pred_sentence = preds[i]

    # 1. Calculate Alignment
    # This aligns the reference and prediction to find exactly which words match
    output = jiwer.process_words(ref_sentence, pred_sentence)

    # 2. Identify Missing Words (Approximate method for quick analysis)
    ref_words = set(ref_sentence.split())
    pred_words = set(pred_sentence.split())

    # Find words present in Reference but missing in Prediction
    missing = list(ref_words - pred_words)
    all_deletions.extend(missing)

    # 3. Analyze Length Bias
    # If diff is negative, the model predicts sentences that are too short
    len_ref = len(ref_sentence.split())
    len_pred = len(pred_sentence.split())
    diff = len_pred - len_ref
    length_diffs.append(diff)

    # 4. Classify the Error Type
    error_type = "Correct"
    if output.wer == 0:
        error_type = "Perfect Match"
    elif len_pred < len_ref:
        error_type = "Deletion (Too Short)"
    elif len_pred > len_ref:
        error_type = "Insertion (Too Long)"
    else:
        error_type = "Substitution (Wrong Word)"

    # Store only the errors for display
    if output.wer > 0:
        error_samples.append({
            "Type": error_type,
            "Reference": ref_sentence,
            "Prediction": pred_sentence,
            "Missing Words": ", ".join(missing) if missing else "-"
        })

# --- Display Results ---

# 1. Top Missed Words
print("\nTop 10 Most Missed Words (Hardest signs for the model):")
missed_counts = Counter(all_deletions).most_common(10)
for word, count in missed_counts:
    print(f"   - '{word}': Missed {count} times")

# 2. Model Bias Analysis
avg_diff = sum(length_diffs) / len(length_diffs)
print(f"\nModel Bias Analysis (Average Length Difference: {avg_diff:.2f}):")
if avg_diff < -0.5:
    print("   Result: The model is inconsistent. It tends to predict fewer words than necessary (High Deletion Rate).")
elif avg_diff > 0.5:
    print("   Result: The model is aggressive. It tends to predict extra words (High Insertion Rate).")
else:
    print("   Result: The model length is balanced. Most errors are likely due to confusing similar signs (Substitution).")

# 3. Show Error Examples Table
df_samples = pd.DataFrame(error_samples)

print("\n--- Examples: Deletion Errors (Prediction is too short) ---")
if not df_samples[df_samples['Type'].str.contains("Deletion")].empty:
    display(df_samples[df_samples['Type'].str.contains("Deletion")].head(5))

print("\n--- Examples: Substitution Errors (Word Confusion) ---")
if not df_samples[df_samples['Type'].str.contains("Substitution")].empty:
    display(df_samples[df_samples['Type'].str.contains("Substitution")].head(5))

--- Detailed Error Analysis ---

Top 10 Most Missed Words (Hardest signs for the model):
   - 'هو': Missed 54 times
   - 'سوال': Missed 47 times
   - 'انا': Missed 40 times
   - 'ولد': Missed 28 times
   - 'ذهاب': Missed 20 times
   - 'ماضي': Missed 18 times
   - 'شخص': Missed 17 times
   - 'سبب': Missed 16 times
   - 'وقت': Missed 15 times
   - 'الان': Missed 13 times

Model Bias Analysis (Average Length Difference: -0.98):
   Result: The model is inconsistent. It tends to predict fewer words than necessary (High Deletion Rate).

--- Examples: Deletion Errors (Prediction is too short) ---


,Type,Reference,Prediction,Missing Words
1,Deletion (Too Short),استفهام هو معلم هو,استفهام استفهام هو,معلم
2,Deletion (Too Short),هو معلم لا انا مدرسه,هو معلم لا انا,مدرسه
5,Deletion (Too Short),استفهام هو صديق مدرسه,استفهام هو مدرسه,صديق
6,Deletion (Too Short),انا اسره رقم ثلاث اشخاص,انا اسره رقم ثلاث,اشخاص
10,Deletion (Too Short),عمر ابن خمسون سن,عمر ابن سن,خمسون



--- Examples: Substitution Errors (Word Confusion) ---


,Type,Reference,Prediction,Missing Words
0,Substitution (Wrong Word),هو معلم لغه اشاره,هو هو لغه اشاره,معلم
7,Substitution (Wrong Word),عمر اربع سن عمر,عمر اربع سن سن,-
8,Substitution (Wrong Word),عمر سن خمسون سن,عمر سن سن سن,خمسون
11,Substitution (Wrong Word),معرفه هو لغه اشاره,معرفه لغه لغه اشاره,هو
15,Substitution (Wrong Word),انا كتابه,انا رغبه,كتابه


## Error Analysis & Discussion

The output above gives two concrete results.

**The hardest signs are the most common words.**

| Word | Missed |
|:---|---:|
| هو | 54 |
| سوال | 47 |
| انا | 40 |
| ولد | 28 |
| ذهاب | 20 |
| ماضي | 18 |
| شخص | 17 |
| سبب | 16 |
| وقت | 15 |
| الان | 13 |

These are not difficult signs. They are function words and pronouns, the most frequent tokens in the corpus, so they appear in more references than anything else and therefore accumulate more misses. The same three words top the Conformer's *most predicted* list for the same reason. Frequency drives both.

There is a real linguistic effect underneath it though: short function words occupy few frames, and in continuous signing they blur into the signs on either side through co-articulation. CTC has little to align them to, and a blank is the cheap answer.

**The model under-generates.** Average length difference is **-0.98**, so predictions run about one word short of the reference. Deletion is this model's characteristic failure.

That number is worth holding next to the Conformer with a Seq2Seq decoder in `03-conformer-seq2seq.ipynb`, which has an average length difference of **+10.86**. Same encoder family, same data, opposite pathology: CTC drops a word, the autoregressive decoder runs away. CTC's monotonic alignment constraint is what keeps the output tied to the input length, and removing it costs more than it gains here.

**What the example sections show.** The two blocks above, for deletion and substitution errors, print their headers and no rows: the selection logic found nothing to display. The per-word counts and the length statistic are the results this analysis actually produced.

## Where this would go next

**Language model integration during beam search.** Decoding currently scores acoustic evidence alone. An Arabic language model would supply the prior that makes a dropped function word expensive, which targets exactly the deletion bias measured above.

**Stronger temporal augmentation.** More aggressive speed variation would give the short, fast signs more varied examples.

**Keypoint selection.** The 86 joints include points that carry little linguistic signal; restricting the input to hands, face and upper body would raise the signal-to-noise ratio per frame.

The most direct answer, though, is the one the project already took: add the convolutional module. The Conformer differs from this model in little else and reaches 13.04%.